# Bước 1: Chuẩn bị Dữ liệu (Data Preparation)
Notebook này thực hiện việc tải tập câu hỏi SQuAD, NQ và trích xuất Corpus thu nhỏ (8,000 bài viết) từ HuggingFace bằng DuckDB.

In [ ]:
!pip install datasets duckdb pandas pyarrow

## 1. Tải 1000 câu hỏi (SQuAD và NQ)

In [ ]:
import random
import pandas as pd
from datasets import load_dataset
import os

cache_folder = '/kaggle/working/hf_cache'
os.makedirs(cache_folder, exist_ok=True)
random.seed(42)

# 1. SQuAD (lấy 500 câu)
print('Loading SQuAD...')
squad = load_dataset('rajpurkar/squad', split='validation', cache_dir=cache_folder)
squad_df = squad.to_pandas()
squad_sample = squad_df.sample(500, random_state=42)
squad_gold_titles = squad_sample['title'].unique().tolist()

# 2. NQ-open (lấy 500 câu)
print('Loading NQ...')
nq_open = load_dataset('google-research-datasets/nq_open', split='validation', cache_dir=cache_folder)
nq_open_df = nq_open.to_pandas()
nq_sample = nq_open_df.sample(500, random_state=42)

# NQ-open không có title, dùng chế độ STREAMING để không bị tràn ổ cứng
print('Loading Natural Questions via Streaming to extract titles...')
nq_full_stream = load_dataset('google-research-datasets/natural_questions', split='validation', streaming=True)

question_to_title = {}
for row in nq_full_stream:
    q_text = row['question']['text']
    t = row['document']['title']
    question_to_title[q_text] = t

# Map để lấy title
nq_sample['title'] = nq_sample['question'].map(question_to_title)
nq_merged = nq_sample.dropna(subset=['title']).head(500)
nq_gold_titles = nq_merged['title'].unique().tolist()

gold_titles = list(set(squad_gold_titles + nq_gold_titles))
print(f'Total Gold Titles: {len(gold_titles)}')

# Lưu tập câu hỏi
test_questions = pd.concat([
    squad_sample[['question', 'answers', 'title']],
    nq_merged[['question', 'answer', 'title']].rename(columns={'answer': 'answers'})
])
test_questions.to_json('/kaggle/working/test_questions.jsonl', orient='records', lines=True)
print('Saved test_questions.jsonl')

## 2. Dùng DuckDB trích xuất Corpus 8000 bài từ HuggingFace

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

jsonl_url = 'hf://datasets/chentong00/factoid-wiki/data/*.jsonl'

print('Đang lấy ngẫu nhiên Distractor Titles...')
num_distractors = 8000 - len(gold_titles)

# FIX: Khai thác chính xác title_span từ trường metadata JSON của dataset.
distractor_query = f"""
SELECT DISTINCT substring(contents, CAST(metadata.title_span[1] AS INTEGER) + 1, CAST(metadata.title_span[2] AS INTEGER) - CAST(metadata.title_span[1] AS INTEGER)) AS title 
FROM read_json_auto('{jsonl_url}', format='newline_delimited')
LIMIT 15000
"""
distractor_df = con.execute(distractor_query).df()

distractor_titles = [t for t in distractor_df['title'].tolist() if t not in gold_titles]
distractor_titles = distractor_titles[:num_distractors]

all_target_titles = gold_titles + distractor_titles
print(f'Total target titles (Gold + Distractors): {len(all_target_titles)}')

con.execute("CREATE TABLE target_titles AS SELECT * FROM unnest(?::VARCHAR[]) AS t(title)", [all_target_titles])

## 3. Tải Dữ liệu Passage, Sentence, Proposition về máy

In [ ]:
urls = {
    'passage': 'hf://datasets/chentong00/factoid-wiki-passage/data/*.jsonl',
    'sentence': 'hf://datasets/chentong00/factoid-wiki-sentence/data/*.jsonl',
    'proposition': 'hf://datasets/chentong00/factoid-wiki/data/*.jsonl'
}

for level, url in urls.items():
    print(f'Đang tải tập {level}...')
    query = f"""
    SELECT * 
    FROM read_json_auto('{url}', format='newline_delimited')
    WHERE substring(contents, CAST(metadata.title_span[1] AS INTEGER) + 1, CAST(metadata.title_span[2] AS INTEGER) - CAST(metadata.title_span[1] AS INTEGER)) IN (SELECT title FROM target_titles)
    """
    df = con.execute(query).df()
    output_file = f'/kaggle/working/corpus_{level}.parquet'
    df.to_parquet(output_file)
    print(f'Đã lưu {output_file} ({len(df)} dòng)')

print('HOÀN TẤT BƯỚC 1!')